<a href="https://colab.research.google.com/github/Ikram0039/Artificial-Intelligence-Lab/blob/main/Lab_5_complete_4_tasks_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from collections import deque
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Define the tree structure
tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H', 'I'],
    'E': ['J'],
    'F': ['K'],
    'G': [],
    'H': [],
    'I': [],
    'J': [],
    'K': []
}

def dls(node, goal, limit, visited_order):
    """Recursive Depth-Limited Search. Returns 'found', 'cutoff' or 'failure'."""
    visited_order.append(node)
    if node == goal:
        return 'found'
    if limit == 0:
        return 'cutoff'

    cutoff_occurred = False
    for child in tree[node]:
        res = dls(child, goal, limit - 1, visited_order)
        if res == 'found':
            return 'found'
        if res == 'cutoff':
            cutoff_occurred = True

    return 'cutoff' if cutoff_occurred else 'failure'

def iddfs_count(start, goal, max_depth=10):
    """Iterative Deepening Search keeping track of cumulative node visits."""
    total_visits = 0
    for depth in range(max_depth + 1):
        order = []
        res = dls(start, goal, depth, order)
        total_visits += len(order)
        print(f"Depth limit {depth}: visited {' '.join(order):<30} -> {res} (Visits in this step: {len(order)})")
        if res == 'found':
            return depth, total_visits
        if res == 'failure':
            return None, total_visits
    return None, total_visits

def bfs_count(start, goal):
    """Breadth-First Search keeping track of visitation order up to the goal."""
    queue = deque([start])
    visited = {start}
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        if node == goal:
            break
        for child in tree[node]:
            if child not in visited:
                visited.add(child)
                queue.append(child)

    return order

# Execution and comparison
print("--- IDDFS Executing ---")
depth_found, iddfs_total = iddfs_count('A', 'J')

print("\n--- BFS Executing ---")
bfs_order = bfs_count('A', 'J')
bfs_total = len(bfs_order)
print(f"BFS Visitation Order: {' '.join(bfs_order)}")

# Explicit comparison and output print
print("\n=============================================")
print("         SEARCH ALGORITHM COMPARISON         ")
print("=============================================")
print(f"IDDFS Total Nodes Visited: {iddfs_total}")
print(f"BFS Total Nodes Visited  : {bfs_total}")
print("---------------------------------------------")
if iddfs_total > bfs_total:
    print(f"BFS was more efficient on this tree by {iddfs_total - bfs_total} node visits due to IDDFS repeating states at earlier levels.")
elif iddfs_total < bfs_total:
    print(f"IDDFS was more efficient on this tree by {bfs_total - iddfs_total} node visits.")
else:
    print("Both algorithms visited the same number of nodes.")
print("=============================================")

# ----- Visualize the tree with node J highlighted -----
G = nx.DiGraph([(p, c) for p, cs in tree.items() for c in cs])
pos, next_x = {}, [0]

def layout(n, d=0):
    kids = tree[n]
    if not kids:
        pos[n] = (next_x[0], -d)
        next_x[0] += 1
    else:
        for k in kids:
            layout(k, d + 1)
        pos[n] = (sum(pos[k][0] for k in kids) / len(kids), -d)

layout('A')
NAVY = '#0E367B'
colors = ['white' if n != 'J' else NAVY for n in G.nodes]

plt.figure(figsize=(6, 3.6))
nx.draw(G, pos, with_labels=False, node_color=colors, node_size=700, edgecolors=NAVY, edge_color=NAVY, arrows=False)
nx.draw_networkx_labels(G, pos, font_weight='bold', font_color={n: 'white' if n == 'J' else NAVY for n in G.nodes})
plt.title("Search tree (goal J highlighted)")
plt.savefig('lab04_tree.png', dpi=130, bbox_inches='tight')
print("\nTree visualization saved to lab04_tree.png")


--- IDDFS Executing ---
Depth limit 0: visited A                              -> cutoff (Visits in this step: 1)
Depth limit 1: visited A B C                          -> cutoff (Visits in this step: 3)
Depth limit 2: visited A B D E C F G                  -> cutoff (Visits in this step: 7)
Depth limit 3: visited A B D H I E J                  -> found (Visits in this step: 7)

--- BFS Executing ---
BFS Visitation Order: A B C D E F G H I J

         SEARCH ALGORITHM COMPARISON         
IDDFS Total Nodes Visited: 18
BFS Total Nodes Visited  : 10
---------------------------------------------
BFS was more efficient on this tree by 8 node visits due to IDDFS repeating states at earlier levels.

Tree visualization saved to lab04_tree.png


Task 2

In [9]:
from collections import deque
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Define the tree structure
tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H', 'I'],
    'E': ['J'],
    'F': ['K'],
    'G': [],
    'H': [],
    'I': [],
    'J': [],
    'K': []
}

def dls(node, goal, limit, visited_order):
    """Recursive Depth-Limited Search. Returns 'found', 'cutoff' or 'failure'."""
    visited_order.append(node)
    if node == goal:
        return 'found'
    if limit == 0:
        return 'cutoff'

    cutoff_occurred = False
    for child in tree[node]:
        res = dls(child, goal, limit - 1, visited_order)
        if res == 'found':
            return 'found'
        if res == 'cutoff':
            cutoff_occurred = True

    return 'cutoff' if cutoff_occurred else 'failure'

def save_depth_visualization(G, pos, visited_nodes, depth_limit):
    """Saves a tree visualization highlighting visited nodes in green and unvisited in grey."""
    # Colors: Green if visited during this depth limit, Grey otherwise
    node_colors = ['#2ca02c' if node in visited_nodes else '#d3d3d3' for node in G.nodes]
    # Label colors: White if visited (for readability), Navy if unvisited
    label_colors = {node: 'white' if node in visited_nodes else '#555555' for node in G.nodes}

    plt.figure(figsize=(6, 4))
    nx.draw(
        G, pos, with_labels=False,
        node_color=node_colors, node_size=700,
        edgecolors='#555555', edge_color='#cccccc', arrows=False
    )
    nx.draw_networkx_labels(G, pos, font_weight='bold', font_color=label_colors)

    plt.title(f"IDDFS Iteration (Depth Limit {depth_limit})\nGreen: Visited | Grey: Unvisited")
    filename = f"iddfs_limit_{depth_limit}.png"
    plt.savefig(filename, dpi=130, bbox_inches='tight')
    plt.close()
    print(f"  Saved search pattern visualization: {filename}")

def iddfs_count(start, goal, G, pos, max_depth=10):
    """Iterative Deepening Search keeping track of cumulative node visits and saving step images."""
    total_visits = 0
    for depth in range(max_depth + 1):
        order = []
        res = dls(start, goal, depth, order)
        total_visits += len(order)
        print(f"Depth limit {depth}: visited {' '.join(order):<30} -> {res} (Visits in this step: {len(order)})")

        # Save search pattern visualization for the current depth limit
        save_depth_visualization(G, pos, set(order), depth)

        if res == 'found':
            return depth, total_visits
        if res == 'failure':
            return None, total_visits
    return None, total_visits

def bfs_count(start, goal):
    """Breadth-First Search keeping track of visitation order up to the goal."""
    queue = deque([start])
    visited = {start}
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        if node == goal:
            break
        for child in tree[node]:
            if child not in visited:
                visited.add(child)
                queue.append(child)

    return order

# ----- Setup Tree Layout for Visualization -----
G = nx.DiGraph([(p, c) for p, cs in tree.items() for c in cs])
pos, next_x = {}, [0]

def layout(n, d=0):
    kids = tree[n]
    if not kids:
        pos[n] = (next_x[0], -d)
        next_x[0] += 1
    else:
        for k in kids:
            layout(k, d + 1)
        pos[n] = (sum(pos[k][0] for k in kids) / len(kids), -d)

layout('A')

# Execution and comparison
print("--- IDDFS Executing & Visualizing ---")
depth_found, iddfs_total = iddfs_count('A', 'J', G, pos)

print("\n--- BFS Executing ---")
bfs_order = bfs_count('A', 'J')
bfs_total = len(bfs_order)
print(f"BFS Visitation Order: {' '.join(bfs_order)}")

# Explicit comparison and output print
print("\n=============================================")
print("         SEARCH ALGORITHM COMPARISON         ")
print("=============================================")
print(f"IDDFS Total Nodes Visited: {iddfs_total}")
print(f"BFS Total Nodes Visited  : {bfs_total}")
print("---------------------------------------------")
if iddfs_total > bfs_total:
    print(f"BFS was more efficient on this tree by {iddfs_total - bfs_total} node visits due to IDDFS repeating states at earlier levels.")
elif iddfs_total < bfs_total:
    print(f"IDDFS was more efficient on this tree by {bfs_total - iddfs_total} node visits.")
else:
    print("Both algorithms visited the same number of nodes.")
print("=============================================")

# ----- Visualize the final tree with node J highlighted -----
NAVY = '#0E367B'
colors = ['white' if n != 'J' else NAVY for n in G.nodes]

plt.figure(figsize=(6, 3.6))
nx.draw(G, pos, with_labels=False, node_color=colors, node_size=700, edgecolors=NAVY, edge_color=NAVY, arrows=False)
nx.draw_networkx_labels(G, pos, font_weight='bold', font_color={n: 'white' if n == 'J' else NAVY for n in G.nodes})
plt.title("Search tree (goal J highlighted)")
plt.savefig('lab04_tree.png', dpi=130, bbox_inches='tight')
plt.close()
print("\nFinal static tree visualization saved to lab04_tree.png")

--- IDDFS Executing & Visualizing ---
Depth limit 0: visited A                              -> cutoff (Visits in this step: 1)
  Saved search pattern visualization: iddfs_limit_0.png
Depth limit 1: visited A B C                          -> cutoff (Visits in this step: 3)
  Saved search pattern visualization: iddfs_limit_1.png
Depth limit 2: visited A B D E C F G                  -> cutoff (Visits in this step: 7)
  Saved search pattern visualization: iddfs_limit_2.png
Depth limit 3: visited A B D H I E J                  -> found (Visits in this step: 7)
  Saved search pattern visualization: iddfs_limit_3.png

--- BFS Executing ---
BFS Visitation Order: A B C D E F G H I J

         SEARCH ALGORITHM COMPARISON         
IDDFS Total Nodes Visited: 18
BFS Total Nodes Visited  : 10
---------------------------------------------
BFS was more efficient on this tree by 8 node visits due to IDDFS repeating states at earlier levels.

Final static tree visualization saved to lab04_tree.png


In [11]:
def get_moves(state):
    # Find the index of the empty tile (0)
    idx = state.index(0)
    row, col = idx // 3, idx % 3
    moves = []

    # Define possible movements: (row_offset, col_offset)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in directions:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            # Swap tiles to create new state
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            moves.append(tuple(new_state))
    return moves

def dls_puzzle(state, goal, limit, visited, expanded_nodes):
    if state == goal:
        return True, [state]
    if limit <= 0:
        return False, []

    # Count state expansion
    expanded_nodes.add(state)

    # Mark state as visited on current path to prevent cycles
    visited.add(state)

    for next_state in get_moves(state):
        if next_state not in visited:
            found, path = dls_puzzle(next_state, goal, limit - 1, visited, expanded_nodes)
            if found:
                return True, [state] + path

    visited.remove(state)
    return False, []

def iddfs_puzzle(start, goal):
    depth = 0
    expanded_nodes = set()
    while True:
        visited = set()
        found, path = dls_puzzle(start, goal, depth, visited, expanded_nodes)
        if found:
            return depth, len(expanded_nodes), path
        depth += 1

# Define Start and Goal states
start_state = (1, 2, 3, 4, 0, 6, 7, 5, 8)
goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)

print("Solving 8-puzzle with IDDFS...")
depth_found, total_expanded, solution_path = iddfs_puzzle(start_state, goal_state)

print("\n=============================================")
print("               IDDFS RESULTS                 ")
print("=============================================")
print(f"Goal Found at Depth       : {depth_found}")
print(f"Total Unique Nodes Expanded: {total_expanded}")
print("=============================================")

Solving 8-puzzle with IDDFS...

               IDDFS RESULTS                 
Goal Found at Depth       : 2
Total Unique Nodes Expanded: 3


Task 3

In [12]:
def get_moves(state):
    # Find the index of the empty tile (0)
    idx = state.index(0)
    row, col = idx // 3, idx % 3
    moves = []

    # Define possible movements: (row_offset, col_offset)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in directions:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr * 3 + nc
            # Swap tiles to create new state
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            moves.append(tuple(new_state))
    return moves

def dls_puzzle(state, goal, limit, visited, expanded_nodes):
    if state == goal:
        return True, [state]
    if limit <= 0:
        return False, []

    # Count state expansion
    expanded_nodes.add(state)

    # Mark state as visited on current path to prevent cycles
    visited.add(state)

    for next_state in get_moves(state):
        if next_state not in visited:
            found, path = dls_puzzle(next_state, goal, limit - 1, visited, expanded_nodes)
            if found:
                return True, [state] + path

    visited.remove(state)
    return False, []

def iddfs_puzzle(start, goal):
    depth = 0
    expanded_nodes = set()
    while True:
        visited = set()
        found, path = dls_puzzle(start, goal, depth, visited, expanded_nodes)
        if found:
            return depth, len(expanded_nodes), path
        depth += 1

# Define Start and Goal states
start_state = (1, 2, 3, 4, 0, 6, 7, 5, 8)
goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)

print("Solving 8-puzzle with IDDFS...")
depth_found, total_expanded, solution_path = iddfs_puzzle(start_state, goal_state)

print("\n=============================================")
print("               IDDFS RESULTS                 ")
print("=============================================")
print(f"Goal Found at Depth       : {depth_found}")
print(f"Total Unique Nodes Expanded: {total_expanded}")
print("=============================================")

Solving 8-puzzle with IDDFS...

               IDDFS RESULTS                 
Goal Found at Depth       : 2
Total Unique Nodes Expanded: 3


Task 4

In [13]:
# Grid maze definition from Lab 3
MAZE = [
    "S . . . . # #",
    "# # . # # . .",
    ". . . # . . #",
    ". # . . . . #",
    ". . . . # . G"
]

grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

def find(ch):
    for r in range(ROWS):
        for c in range(COLS):
            if grid[r][c] == ch:
                return (r, c)

def neighbours(cell):
    r, c = cell
    # Up, Right, Down, Left movements
    for dr, dc in [(-1, 0), (0, 1), (1, 0), (0, -1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < ROWS and 0 <= nc < COLS and grid[nr][nc] != '#':
            yield (nr, nc)

def dls(node, goal, limit, visited):
    if node == goal:
        return 'found'
    if limit == 0:
        return 'cutoff'

    visited.add(node)
    any_cutoff = False

    for nb in neighbours(node):
        if nb not in visited:
            res = dls(nb, goal, limit - 1, visited)
            if res == 'found':
                return 'found'
            elif res == 'cutoff':
                any_cutoff = True

    visited.remove(node) # Backtrack
    return 'cutoff' if any_cutoff else 'failure'

S = find('S')
G = find('G')

for limit in [5, 10, 15]:
    visited = set()
    result = dls(S, G, limit, visited)
    print(f"Depth Limit: {limit:<2} -> Result: {result.upper()}")

Depth Limit: 5  -> Result: CUTOFF
Depth Limit: 10 -> Result: FOUND
Depth Limit: 15 -> Result: FOUND
